In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install transformers scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [5]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
with open("train.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open("test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

In [7]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["reviews"].tolist(),
    train_df["sentiments"].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=train_df["sentiments"].tolist()
)

In [8]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=256):
        self.texts = [str(t).strip() for t in texts]
        self.labels = labels
        self.encodings = tokenizer(
            self.texts,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "raw_text": self.texts[idx]
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = ReviewDataset(train_texts, train_labels, MAX_LEN)
val_dataset = ReviewDataset(val_texts, val_labels, MAX_LEN)
test_dataset = ReviewDataset(test_df["reviews"].tolist(), None, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [9]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from tqdm.auto import tqdm

# 1. Optimize thread allocation if executing on CPU
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 4)

best_val_f1 = 0.0

# 2. Training and validation loop with live progress bars
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})

    # Validation step
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(batch["labels"].numpy())

    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro")
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {total_train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f} | Val Macro F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_distilbert.pt")

Epoch 1/4 [Train]:   0%|          | 0/394 [00:00<?, ?it/s]

Epoch 1/4 [Val]:   0%|          | 0/70 [00:00<?, ?it/s]

Epoch 1/4 | Train Loss: 0.2784 | Val Acc: 0.9352 | Val Macro F1: 0.8650


Epoch 2/4 [Train]:   0%|          | 0/394 [00:00<?, ?it/s]

Epoch 2/4 [Val]:   0%|          | 0/70 [00:00<?, ?it/s]

Epoch 2/4 | Train Loss: 0.1159 | Val Acc: 0.9424 | Val Macro F1: 0.8738


... (2 outputs left out)

Epoch 3/4 | Train Loss: 0.0530 | Val Acc: 0.9469 | Val Macro F1: 0.8915


Epoch 4/4 [Train]:   0%|          | 0/394 [00:00<?, ?it/s]

Epoch 4/4 [Val]:   0%|          | 0/70 [00:00<?, ?it/s]

Epoch 4/4 | Train Loss: 0.0253 | Val Acc: 0.9487 | Val Macro F1: 0.8951


In [11]:
model.load_state_dict(torch.load("best_distilbert.pt"))
model.eval()
test_predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        test_predictions.extend(preds)

submission_df = pd.DataFrame({
    "reviews": test_df["reviews"],
    "sentiments": test_predictions
})
submission_df.to_csv("submission.csv", index=False)
print("Saved test predictions to submission.csv")

Saved test predictions to submission.csv
